In [ ]:
# step 1 # load and explore the dataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import seaborn as sns
import torch
import torch.nn.functional as F
import random


In [ ]:
def load_triples(path):
    triples = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                triples.append(tuple(parts))
    return triples

train_triples = load_triples("../data/train.txt")
test_triples = load_triples("../data/test.txt")


### Entity and Relation Extraction

This cell extracts all **unique entities** and **relations** from the training data to build vocabularies:

- **Entities**: Combines all heads and tails from triples (people in the family graph)
- **Relations**: Extracts all unique relationship types (e.g., "motherOf", "fatherOf")
- **Mapping Dictionaries**: Creates `ent2id` and `rel2id` to convert text names to integer IDs
  - Required for embedding lookup in neural networks
  - Each entity/relation gets a unique index

This is the **first step** in preparing the knowledge graph for embedding-based learning.

In [ ]:
entities = sorted(
    {h for h, r, t in train_triples} |
    {t for h, r, t in train_triples}
)

relations = sorted({r for h, r, t in train_triples})

ent2id = {e: i for i, e in enumerate(entities)}
rel2id = {r: i for i, r in enumerate(relations)}


### Converting Triples to Tensor Format

This cell transforms text-based triples into **numerical tensors** suitable for PyTorch:

- **Input**: Triples like `("Alice", "motherOf", "Bob")`
- **Output**: Tensor with integer IDs like `[[25, 3, 47], ...]`
- **Process**: Uses `ent2id` and `rel2id` mappings to convert names to indices

This conversion is essential because:
- Neural networks operate on numbers, not strings
- Tensors enable efficient batch processing on GPU
- Integer IDs allow direct embedding lookup

Both **train_tensor** and **test_tensor** are created for model training and evaluation.

In [ ]:
def triples_to_tensor(triples, ent2id, rel2id):
    return torch.tensor(
        [[ent2id[h], rel2id[r], ent2id[t]] for h, r, t in triples],
        dtype=torch.long
    )

train_tensor = triples_to_tensor(train_triples, ent2id, rel2id)
test_tensor = triples_to_tensor(test_triples, ent2id, rel2id)

### Initializing ComplEx Embeddings

This cell creates the **embedding matrices** for the ComplEx (Complex Embeddings) model:

**ComplEx Model**: Represents entities and relations in **complex vector space** (real + imaginary parts)

- **ent_re, ent_im**: Real and imaginary embeddings for entities (shape: `[num_entities, 200]`)
- **rel_re, rel_im**: Real and imaginary embeddings for relations (shape: `[num_relations, 200]`)
- **embedding_dim = 200**: Each entity/relation has a 200-dimensional complex vector
- **requires_grad=True**: Enables gradient computation for learning via backpropagation

**Why ComplEx?** Complex numbers allow modeling **antisymmetric** relations (e.g., "childOf" vs "parentOf") more effectively than real-valued embeddings.

These embeddings will be **learned during training** to capture the structure of the knowledge graph.

In [ ]:
num_entities = len(ent2id)
num_relations = len(rel2id)
embedding_dim = 200

ent_re = torch.randn(num_entities, embedding_dim, requires_grad=True)
ent_im = torch.randn(num_entities, embedding_dim, requires_grad=True)

rel_re = torch.randn(num_relations, embedding_dim, requires_grad=True)
rel_im = torch.randn(num_relations, embedding_dim, requires_grad=True)


### ComplEx Scoring Function

This is the **core scoring function** that computes how plausible a triple (h, r, t) is:

**ComplEx Formula**: The score is computed using complex number multiplication:
```
score = Re(⟨h, r, t̄⟩)
```

Expanded to:
```
score = h_re * r_re * t_re + h_im * r_re * t_im + h_re * r_im * t_im - h_im * r_im * t_re
```

**Intuition**:
- **Higher score** = triple is more likely to be true
- **Lower score** = triple is less plausible
- The function captures interactions between real and imaginary components
- Complex arithmetic naturally handles symmetric and antisymmetric relations

**Purpose**: Used to score both **positive triples** (from data) and **negative triples** (corrupted) during training.

In [ ]:
def complex_score(h, r, t, ent_re, ent_im, rel_re, rel_im):
    h_re, h_im = ent_re[h], ent_im[h]
    r_re, r_im = rel_re[r], rel_im[r]
    t_re, t_im = ent_re[t], ent_im[t]

    score = (
        h_re * r_re * t_re +
        h_im * r_re * t_im +
        h_re * r_im * t_im -
        h_im * r_im * t_re
    ).sum(dim=1)

    return score



### Negative Sampling for Training

This function generates **negative (false) triples** by corrupting real triples:

**Strategy**: For each triple `(h, r, t)`:
- With 50% probability: Replace the **head** entity with a random entity
- With 50% probability: Replace the **tail** entity with a random entity

**Purpose in Training**:
- The model learns by **contrasting** true triples (positive samples) with false triples (negative samples)
- Positive triples should get **high scores**, negative triples should get **low scores**
- This teaches the model to distinguish valid from invalid relations

**Example**:
- Positive: `(Alice, motherOf, Bob)` → should score high
- Negative: `(Alice, motherOf, Charlie)` → should score low (if Charlie is not Alice's child)

This is a standard technique in **knowledge graph embedding** for contrastive learning.

In [ ]:
def negative_sampling(triples, num_entities):
    neg = triples.clone()
    for i in range(len(neg)):
        if random.random() < 0.5:
            neg[i, 0] = random.randint(0, num_entities - 1)
        else:
            neg[i, 2] = random.randint(0, num_entities - 1)
    return neg


### Training the ComplEx Model

This cell implements the **main training loop** to learn entity and relation embeddings:

**Key Components**:

1. **Optimizer**: Adam optimizer with learning rate 0.002 to update all embeddings
2. **Training Setup**: 350 epochs, batch size of 256 triples per update
3. **Training Process** (per epoch):
   - Shuffle training data randomly
   - For each mini-batch:
     - Sample **negative triples** via corruption
     - Compute scores for **positive** and **negative** triples
     - Calculate **loss**: 
       - Positive triples should have scores close to 1
       - Negative triples should have scores close to 0
       - Uses binary cross-entropy loss
     - **Backpropagate** gradients and update embeddings

4. **Loss Tracking**: Records total loss per epoch for visualization

**Goal**: After training, embeddings should capture the **semantic structure** of the knowledge graph, enabling link prediction and relation inference.

In [ ]:
optimizer = torch.optim.Adam(
    [ent_re, ent_im, rel_re, rel_im],
    lr=0.002
)

epochs = 350
batch_size = 256
losses = []

for epoch in range(epochs):
    perm = torch.randperm(train_tensor.size(0))
    epoch_loss = 0.0

    for i in range(0, train_tensor.size(0), batch_size):
        batch = train_tensor[perm[i:i + batch_size]]

        neg_batch = negative_sampling(batch, num_entities)

        pos_score = complex_score(batch[:, 0],batch[:, 1],batch[:, 2],ent_re, ent_im, rel_re, rel_im)

        neg_score = complex_score(neg_batch[:, 0],neg_batch[:, 1],neg_batch[:, 2],ent_re, ent_im, rel_re, rel_im)

        loss = (
            F.binary_cross_entropy_with_logits(
                pos_score, torch.ones_like(pos_score)
            ) +
            F.binary_cross_entropy_with_logits(
                neg_score, torch.zeros_like(neg_score)
            )
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")


### Link Prediction Evaluation

This function evaluates how well the trained model can **predict missing links** in the knowledge graph:

**Task**: Given a query `(head, relation, ?)`, rank all possible tail entities and find the correct one

**Process**:
1. For each test triple `(h, r, t_true)`:
   - Score the head and relation against **ALL possible tail entities**
   - Rank entities by score (higher = more likely)
   - Find the **rank** of the true tail entity

2. **Metrics**:
   - **MRR (Mean Reciprocal Rank)**: Average of 1/rank
     - Higher is better (1.0 = perfect)
     - Emphasizes getting the correct answer in top positions
   - **Hits@1**: % of times the correct entity is ranked 1st
   - **Hits@10**: % of times the correct entity is in top 10

**Interpretation**:
- High MRR/Hits → Model learned good embeddings for link prediction
- These metrics evaluate the model's ability to infer missing family relationships

**Example**: If asked "Who is Alice's child?", can the model rank Bob highest?

In [ ]:
def evaluate_link_prediction(
    test_triples,
    ent_re, ent_im,
    rel_re, rel_im,
    num_entities
):
    ranks = []

    with torch.no_grad():
        for h, r, t in test_triples:
            h = h.unsqueeze(0)
            r = r.unsqueeze(0)

            candidates = torch.arange(num_entities)

            h_expand = h.repeat(num_entities)
            r_expand = r.repeat(num_entities)

            scores = complex_score(
                h_expand, r_expand, candidates,
                ent_re, ent_im, rel_re, rel_im
            )

            # Higher score = more likely
            sorted_indices = torch.argsort(scores, descending=True)

            rank = (sorted_indices == t).nonzero(as_tuple=True)[0].item() + 1
            ranks.append(rank)

    ranks = torch.tensor(ranks, dtype=torch.float)

    mrr = torch.mean(1.0 / ranks).item()
    hits1 = torch.mean((ranks <= 1).float()).item()
    hits10 = torch.mean((ranks <= 10).float()).item()

    return mrr, hits1, hits10


In [ ]:
mrr, hits1, hits10 = evaluate_link_prediction(
    test_tensor,
    ent_re, ent_im,
    rel_re, rel_im,
    num_entities=len(ent2id)
)

print(f"MRR    : {mrr:.4f}")
print(f"Hits@1 : {hits1:.4f}")
print(f"Hits@10: {hits10:.4f}")


In [ ]:
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ComplEx Training Loss")
plt.show()


## Step 5: Making Predictions with the Trained Model

Now let's use the trained ComplEx model to make actual predictions on the knowledge graph. We'll demonstrate:
- Predicting missing tails: Given (head, relation, ?), find the most likely tail entities
- Showing top-K predictions with confidence scores
- Testing on real family relationships from our data

In [ ]:
def predict_tail(head_name, relation_name, top_k=10):
    """
    Predict the most likely tail entities for a given (head, relation) query
    
    Args:
        head_name: Name of the head entity
        relation_name: Name of the relation
        top_k: Number of top predictions to return
    
    Returns:
        List of (entity_name, score) tuples
    """
    if head_name not in ent2id:
        return f"Entity '{head_name}' not found in the knowledge graph"
    if relation_name not in rel2id:
        return f"Relation '{relation_name}' not found in the knowledge graph"
    
    h_idx = ent2id[head_name]
    r_idx = rel2id[relation_name]
    
    # Create tensors for all possible tails
    h = torch.tensor([h_idx])
    r = torch.tensor([r_idx])
    candidates = torch.arange(num_entities)
    
    # Expand head and relation to match all candidates
    h_expand = h.repeat(num_entities)
    r_expand = r.repeat(num_entities)
    
    # Score all possible tails
    with torch.no_grad():
        scores = complex_score(
            h_expand, r_expand, candidates,
            ent_re, ent_im, rel_re, rel_im
        )
    
    # Get top-k predictions
    top_scores, top_indices = torch.topk(scores, k=min(top_k, num_entities))
    
    # Convert back to entity names
    predictions = []
    for score, idx in zip(top_scores, top_indices):
        entity_name = entities[idx.item()]
        predictions.append((entity_name, score.item()))
    
    return predictions

print("Prediction function created successfully!")

### Example 1: Predicting Children (motherOf relation)

Let's find a person from our dataset and predict who their children might be.

In [ ]:
# Find someone who has motherOf relations in the training data
sample_mothers = [h for h, r, t in train_triples if r == 'motherOf'][:5]

print("Sample people with 'motherOf' relations:")
for mother in sample_mothers:
    print(f"  - {mother}")

# Pick one example
if sample_mothers:
    example_person = sample_mothers[0]
    
    print(f"Query: Who are {example_person}'s children?")
    print(f"(Finding: {example_person} --motherOf--> ?)")
    
    
    predictions = predict_tail(example_person, 'motherOf', top_k=10)
    
    # Get actual children from training data for comparison
    actual_children = [t for h, r, t in train_triples if h == example_person and r == 'motherOf']
    
    print(f"Top 10 Predictions (with scores):")
    for i, (entity, score) in enumerate(predictions, 1):
        marker = "✓ CORRECT" if entity in actual_children else ""
        print(f"  {i:2d}. {entity:40s} | Score: {score:8.4f} {marker}")
    
    print(f"\nActual children from training data: {', '.join(actual_children)}")
else:
    print("No motherOf relations found in training data")

### Example 2: Predicting Parents (daughterOf relation)

Now let's predict who someone's parent is.

In [ ]:
# Find someone who has daughterOf relations
if 'daughterOf' in rel2id:
    sample_daughters = [h for h, r, t in train_triples if r == 'daughterOf'][:3]
    
    if sample_daughters:
        example_daughter = sample_daughters[0]
        
        print(f"Query: Who is {example_daughter}'s parent?")
        print(f"(Finding: {example_daughter} --daughterOf--> ?)")
       
        
        predictions = predict_tail(example_daughter, 'daughterOf', top_k=10)
        
        # Get actual parent
        actual_parents = [t for h, r, t in train_triples if h == example_daughter and r == 'daughterOf']
        
        print(f"Top 10 Predictions (with scores):")
        for i, (entity, score) in enumerate(predictions, 1):
            marker = "✓ CORRECT" if entity in actual_parents else ""
            print(f"  {i:2d}. {entity:40s} | Score: {score:8.4f} {marker}")
        
        print(f"\nActual parent from training data: {', '.join(actual_parents)}")
    else:
        print("No daughterOf relations found")
else:
    print("daughterOf relation not in dataset")

### Example 3: Predicting Grandchildren (grandmotherOf relation)

Testing on multi-generational relationships.

In [ ]:
# Find someone who has grandmotherOf relations
if 'grandmotherOf' in rel2id:
    sample_grandmothers = [h for h, r, t in train_triples if r == 'grandmotherOf'][:3]
    
    if sample_grandmothers:
        example_grandmother = sample_grandmothers[0]
       
        print(f"Query: Who are {example_grandmother}'s grandchildren?")
        print(f"(Finding: {example_grandmother} --grandmotherOf--> ?)")
       
        
        predictions = predict_tail(example_grandmother, 'grandmotherOf', top_k=10)
        
        # Get actual grandchildren
        actual_grandchildren = [t for h, r, t in train_triples if h == example_grandmother and r == 'grandmotherOf']
        
        print(f"Top 10 Predictions (with scores):")
        for i, (entity, score) in enumerate(predictions, 1):
            marker = "✓ CORRECT" if entity in actual_grandchildren else ""
            print(f"  {i:2d}. {entity:40s} | Score: {score:8.4f} {marker}")
        
        print(f"\nActual grandchildren from training data: {', '.join(actual_grandchildren[:10])}")
        if len(actual_grandchildren) > 10:
            print(f"  ... and {len(actual_grandchildren) - 10} more")
    else:
        print("No grandmotherOf relations found")
else:
    print("grandmotherOf relation not in dataset")

### Make Your Own Predictions

You can use the `predict_tail()` function to make predictions for any entity and relation in the knowledge graph. 

**Usage**: `predict_tail(entity_name, relation_name, top_k=10)`

In [ ]:
# Show available relations
print("Available Relations in the Knowledge Graph:")

for i, rel in enumerate(relations, 1):
    count = len([1 for h, r, t in train_triples if r == rel])
    print(f"  {i:2d}. {rel:30s} ({count:4d} instances)")

print(f"\n{'=' * 70}")
print(f"Total entities: {len(entities)}")
print(f"Total relations: {len(relations)}")
print(f"\nSample entities: {entities[:10]}")
print(f"... and {len(entities) - 10} more entities")

In [ ]:
# Example: Make a custom prediction
# Replace these with any entity and relation from above
custom_entity = entities[10]  # Pick any entity
custom_relation = relations[0]  # Pick any relation

print(f"Custom Query: {custom_entity} --{custom_relation}--> ?")
predictions = predict_tail(custom_entity, custom_relation, top_k=5)

if isinstance(predictions, str):
    print(predictions)  # Error message
else:
    print(f"\nTop 5 Predictions:")
    for i, (entity, score) in enumerate(predictions, 1):
        print(f"  {i}. {entity:40s} | Score: {score:8.4f}")

### Summary: Model Performance Analysis

**What the predictions show:**

1. **High Scores for Correct Entities**: When the model assigns high scores (closer to 0 or positive values) to entities, those are the most confident predictions. The correct entities from training data should ideally appear at the top of the rankings.

2. **Score Interpretation**: 
   - Scores are computed using the ComplEx formula on learned embeddings
   - Higher scores indicate stronger predicted relationships
   - The model learned these patterns from the training.

3. **Model Capabilities**:
   - Can predict direct relationships (parent-child)
   - Can handle multi-generational relationships (grandparent-grandchild)
   - Can distinguish between different relation types
   - Rankings help identify multiple plausible candidates


In [ ]:
# Visualize prediction scores for a sample query
if sample_mothers:
    example_person = sample_mothers[0]
    predictions = predict_tail(example_person, 'motherOf', top_k=20)
    
    # Extract entities and scores
    pred_entities = [p[0] for p in predictions]
    pred_scores = [p[1] for p in predictions]
    
    # Get actual children for highlighting
    actual_children = [t for h, r, t in train_triples if h == example_person and r == 'motherOf']
    colors_viz = ['green' if e in actual_children else 'skyblue' for e in pred_entities]
    
    # Create plot
    plt.figure(figsize=(12, 6))
    bars = plt.barh(range(len(pred_scores)), pred_scores, color=colors_viz, alpha=0.8, edgecolor='black')
    plt.yticks(range(len(pred_scores)), [e[:30] for e in pred_entities], fontsize=8)
    plt.xlabel('ComplEx Score', fontsize=11, fontweight='bold')
    plt.ylabel('Predicted Entities', fontsize=11, fontweight='bold')
    plt.title(f'Prediction Scores: {example_person} --motherOf--> ?\n(Green = Correct from training data)', 
              fontsize=12, fontweight='bold')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.gca().invert_yaxis()  # Highest score at top
    plt.show()
    
    print(f"Visualization shows top 20 predictions:")
    print(f"  Green bars = Correct predictions (in training data)")
    print(f"  Blue bars = Other predictions")
    print(f"  Higher scores = More confident predictions")